# BPSD v2 to TiLiA

The full run: downloads the BPSD archive, converts the 32 first movements in 11 performances each (352 files), loads the four recordings that BPSD offers freely, builds the package and prints the totals.

Set `DATASET_CONVERTERS_OUT` to the output folder. Run it from the repository root (or with the root on `PYTHONPATH`) with TiLiA's command line in `TILIA`.

In [ ]:
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parent if Path.cwd().name == 'bpsd' else Path.cwd().resolve()))

from bpsd import convert

out = Path(os.environ['DATASET_CONVERTERS_OUT'])
out.mkdir(parents=True, exist_ok=True)

## Download

`fetch` checks the md5 of the whole archive.

In [ ]:
archive = convert.download_archive()
archive

In [ ]:
source = convert.ZipSource(archive)

## Convert every item and run TiLiA

The four performances that BPSD offers freely (AS35, FG58, FJ62, WK64) load their recording from the archive; the others take their length from the annotations.

In [ ]:
summary = convert.convert(convert.ITEMS, out, source=source, audio_source=source)
len(summary)

## Package (no audio, no file paths)

In [ ]:
package = convert.package(out, out / 'package')
package

## Totals

In [ ]:
layers = ['Measures', 'Structure', 'Harmony/keys', 'Harmony/chords']
for layer in layers:
    components = sum(v[layer]['components'] for v in summary.values())
    rows = sum(v[layer]['source_rows'] for v in summary.values())
    print(f'{layer:16} components {components:7}  source rows {rows:7}')
mismatch = [(i, l) for i, v in summary.items() for l, c in v.items() if c['components'] != c['source_rows']]
print(len(summary), 'files;', len(mismatch), 'layers where the file differs from the source')